In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Mapping data from Gotze et al

In [8]:
#Read the data
el_data = pd.read_excel('../data/external/elemental_data_gotze.xlsx', sheet_name='all Means')
el_data_std = pd.read_excel('../data/external/elemental_data_gotze.xlsx', sheet_name='all STDEV')
sample_data = pd.read_csv('../data/external/country_sample.csv')

The groupings are different for each stream. 

Food Waste:  We cannot group the entire organic waste category, so we group it by 'Material type' - food waste and gardening waste <br>
Metal waste: We have to separate ferrous and non-ferrous material types so also grouped by 'Fraction name' <br>
Glass, paper, plastic: they are their own category (in material type) as 'textiles, leather, rubber', 'wood' <br>
Textile and wood are specfic fractions within combustibles <br>
There is no bio plastic category <br>
OTHER is not clear!! - other from combustibles/inert? or all residual waste? <br>

In [9]:
#Melt the table
units = el_data.iloc[0]
el_data.drop(0, inplace=True)
el_data = el_data.melt(id_vars=['Fraction ID', 'Material type', 'Fraction name', 'Waste flow'], var_name='Physiochemical property', value_name='Value')
el_data['Unit'] = el_data['Physiochemical property'].map(units)

In [10]:
#Replace string values with NaN
el_data.loc[el_data['Value'].apply(type) == str,'Value']=np.nan

In [31]:
# Group by for textile and wood
text_wood = el_data.loc[el_data['Fraction name'].isin(['textiles, leather, rubber', 'wood'])].groupby(['Fraction ID','Material type','Fraction name', 'Waste flow', 'Physiochemical property', 'Unit'], as_index = False, dropna=True)['Value'].agg(['min','max'])


In [16]:
#metal fraction names
nfer = ['metal packaging-non-ferrous', 'aluminium foil', 'non-packaging metal-non-ferrous']
fer = ['metal packaging-ferrous', 'non-packaging metal-ferrous']

In [7]:
el_data.columns

Index(['Fraction ID', 'Material type', 'Fraction name', 'Waste flow',
       'Physiochemical property', 'Value', 'Unit'],
      dtype='object')

In [28]:
# Group by for textile and wood
nfmetal = el_data.loc[el_data['Fraction name'].isin(nfer)].groupby(['Material type', 'Waste flow', 'Physiochemical property', 'Unit'], as_index = False, dropna = True)['Value'].agg(['min','max'])
nfmetal['Material type']= 'non-ferrous metal'
fmetal = el_data.loc[el_data['Fraction name'].isin(fer)].groupby(['Material type', 'Waste flow', 'Physiochemical property', 'Unit'], as_index = False)['Value'].agg(['min','max'])
fmetal['Material type']= 'ferrous metal'

In [29]:
metal = pd.concat([nfmetal, fmetal], ignore_index=True)

In [32]:

rest = el_data.loc[(~el_data['Material type'].isin(['combustibles', 'inert ', 'metal']))].groupby(['Material type', 'Waste flow', 'Physiochemical property', 'Unit'], as_index = False, dropna=True)['Value'].agg(['min','max'])

In [33]:
# reshaping and concatenating the dataframes
text_wood.drop(['Fraction ID','Material type'], inplace=True, axis=1)
text_wood.rename(columns={'Fraction name':'Material type'}, inplace=True)


In [37]:
el_data_mapped = pd.concat([rest, text_wood, metal]).reset_index(drop=True)

In [43]:
el_data_mapped['min']=el_data_mapped['min'].astype(float)
el_data_mapped['max']=el_data_mapped['max'].astype(float)

In [45]:
el_data_mapped['Material type'].unique()

array(['food waste', 'gardening waste', 'glass', 'paper & cardboard',
       'plastic', 'textiles, leather, rubber', 'wood',
       'non-ferrous metal', 'ferrous metal'], dtype=object)

In [46]:
code_mapping={
    'bio plastic':'MSW_URB_BIO_PLA', 
    'food waste': 'MSW_URB_FOOD', 
    'glass': 'MSW_URB_GLA', 
    'non-ferrous metal': 'MSW_URB_MET',
    'ferrous metal': 'MSW_URB_MET',
    'other': 'MSW_URB_OTH',
    'paper & cardboard': 'MSW_URB_PAP', 
    'plastic': 'MSW_URB_PLA', 
    'textiles, leather, rubber': 'MSW_URB_TEX',
    'wood': 'MSW_URB_WOOD'
}

In [47]:
el_data_mapped['sector'] = el_data_mapped['Material type'].map(code_mapping)

In [44]:
el_data_mapped['diff']=(el_data_mapped['max']-el_data_mapped['min'])/el_data_mapped['min'].replace(0, np.nan)

In [48]:
#el_data_mapped.to_csv('../data/processed/elemental_data_gotze.csv', index=False)